# Assignment 3, Track C Tutorial: Pure-Prompt RSI (zero training)

**No fine-tuning in this track** — the target model stays frozen from start to finish. You
improve only the **harness**: the instruction, the few-shot demonstrations, and (in the last
round) an **LLM agent that rewrites the prompt for you**. A free Colab T4 is enough
(~1 GPU-hour total); the only external thing is a **free-tier improver API key**
(DeepSeek / Groq / Gemini), used a handful of times.

The prompt-writing job evolves across rounds:

| Round | Who writes the prompt | Called |
|---|---|---|
| **R0** | nobody (fixed template) | baseline |
| **R1** | **you** | human-in-the-loop |
| **R2** | **an LLM agent** that reads dev failures and proposes new configs | agent-in-the-loop |
| **Final** | — | one-shot evaluation on the hidden test set |

This is the "harness" axis of [RSI-Exam](https://rsi-exam.ai) in miniature: iterate on the
**visible** dev split, then run the final artifact **once** on the **hidden** test split.

**Looking for ideas beyond R2?** The real RSI-Exam's public task bank
([rsi-exam.ai/tasks.html](https://rsi-exam.ai/tasks.html)) includes dedicated harness-design
tasks, e.g. *Mathematical and Statistical Reasoning Workflow Design* and *Scientific
Discovery Agent Harness Design* --- exactly this track's game (a frozen model, an improved
inference harness, one-shot hidden-set scoring) at research scale.


## 0. Setup

**Colab (recommended):** upload this notebook (File → Upload notebook) and set
Runtime → Change runtime type → **T4 GPU**.

**Local:** any machine with a CUDA GPU works the same; export your improver key first
(`export IMPROVER_API_KEY=...`) since there are no Colab Secrets locally.


In [ ]:
!pip install -q -U transformers datasets accelerate openai

In [ ]:
import torch

!nvidia-smi
print(torch.__version__, torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Please enable a GPU runtime (Runtime -> Change runtime type -> T4 GPU)" 

## 1. Load the frozen target model

We use **Qwen2.5-0.5B-Instruct** (instruction-tuned, because in this track nothing will be
trained — the model must already follow instructions). It is loaded once and **never updated**.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # frozen for the whole assignment

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to("cuda")
model.eval()   # frozen: we only ever call generate, never train
print(f"#params = {sum(p.numel() for p in model.parameters())/1e9:.2f}B")

## 2. Data: GSM8K and the RSI splits

Same protocol as Track A:

- **Visible** (`train_pool` + `dev`): yours to iterate on. In this track `train_pool` is not
  used for training — 20 of its examples form the **demo pool** you may pick from.
- **Hidden** (`test`): touched **exactly once**, at the very end (Section 7).

In [ ]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")
train_full = gsm8k["train"].shuffle(seed=42)
test_full  = gsm8k["test"].shuffle(seed=42)

DEV_SIZE, TEST_SIZE, DEMO_POOL_SIZE = 100, 100, 20

dev        = train_full.select(range(DEV_SIZE))
demo_pool  = train_full.select(range(DEV_SIZE, DEV_SIZE + DEMO_POOL_SIZE))
test       = test_full.select(range(TEST_SIZE))

print(f"dev={len(dev)}  demo_pool={len(demo_pool)}  test={len(test)}")

## 3. The evaluation harness

The harness = prompt template + demos + decoding + answer extraction. It is the *only* thing
this track is allowed to change. A config is a pair: `{"instruction": str, "demo_ids": tuple}`.

In [ ]:
import re
from tqdm.auto import tqdm
from collections import Counter

INSTR = ("Solve the following math problem. Reason step by step, "
         "then give the final numeric answer on a new line in the format '#### <answer>'.\n\n")

def clean_solution(answer: str) -> str:
    """Strip GSM8K calculator annotations <<...>> from a CoT solution."""
    return re.sub(r"<<[^>]*>>", "", answer)

def extract_answer(text: str) -> str:
    m = re.findall(r"####\s*([\d,\.\-]+)", text)
    if m:
        return m[-1].replace(",", "").strip()
    nums = re.findall(r"-?[\d,]*\.?\d+", text.replace(",", ""))
    return nums[-1].strip() if nums else ""

def normalize(ans):
    try:
        return round(float(ans), 2)
    except (ValueError, TypeError):
        return None

def gold_answer(raw_answer: str):
    return normalize(raw_answer.split("####")[-1].replace(",", ""))

def format_demo(ex):
    cot = clean_solution(ex["answer"].split("####")[0]).strip()
    ans = ex["answer"].split("####")[-1].strip().replace(",", "")
    return f"Problem: {ex['question']}\nSolution: {cot}\n#### {ans}"

def prompt_builder_from_config(cfg):
    """Turn a harness config into a prompt function."""
    demos, instr = [demo_pool[i] for i in cfg["demo_ids"]], cfg["instruction"]
    def builder(question):
        parts = [instr] + [format_demo(d) for d in demos] + [f"Problem: {question}\nSolution:"]
        return "\n\n".join(parts)
    return builder

In [ ]:
def render_prompt(question, builder):
    """Wrap the config-built prompt in the model's chat template (Instruct model)."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": builder(question)}],
        tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_batch(questions, cfg, batch_size=25, max_new_tokens=256, desc="eval"):
    builder = prompt_builder_from_config(cfg)
    texts = [render_prompt(q, builder) for q in questions]
    gen_texts = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False):
        enc = tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True,
                        add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[:, enc["input_ids"].shape[1]:]
        gen_texts += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return gen_texts

def evaluate(cfg, split, batch_size=25, max_new_tokens=256, desc="eval"):
    preds = generate_batch(split["question"], cfg, batch_size=batch_size,
                           max_new_tokens=max_new_tokens, desc=desc)
    records, correct = [], 0
    for q, resp, raw in zip(split["question"], preds, split["answer"]):
        gold, pred = gold_answer(raw), normalize(extract_answer(resp))
        ok = pred is not None and gold is not None and abs(pred - gold) < 1e-4
        correct += ok
        records.append({"question": q, "response": resp, "gold": gold, "pred": pred, "correct": ok})
    return correct / len(records), records

## 4. Round 0 — Zero-shot baseline

The fixed template from Section 3, no demos. Starting point of your RSI curve
(expect roughly 45–60%: the Instruct model already knows the format).

In [ ]:
CFG_R0 = {"instruction": INSTR.strip(), "demo_ids": ()}

r0_dev, r0_records = evaluate(CFG_R0, dev, desc="R0 dev")
print(f"R0 (zero-shot) dev accuracy: {r0_dev:.2%}")

rsi_dev = {"R0 zero-shot": r0_dev}

In [ ]:
# Error analysis first: what kind of failures do you see?
fails = [r for r in r0_records if not r["correct"]]
print(f"{len(fails)} failures on dev. One example:\n")
print("Q:", fails[0]["question"])
print("\nModel output (first 300 chars):\n", fails[0]["response"][:300])
print("\nGold:", fails[0]["gold"], " Extracted:", fails[0]["pred"])

## 5. Round 1 — *Human-in-the-loop*: you improve the harness

Look at the failures above, then hand-craft a better config: a clearer instruction plus one or
two **few-shot demonstrations** from the demo pool (indices `0`–`19` — print a few with the
cell below to choose ones that contrast with the observed failure modes, e.g. multi-step
arithmetic if that is what fails).

In [ ]:
# Browse the demo pool to pick from
for i in [0, 1, 2]:
    print(f"--- demo {i} ---")
    print(format_demo(demo_pool[i])[:400], "\n")

In [ ]:
# YOUR choices here (this is the human-in-the-loop round -- change them and rerun!)
CFG_R1 = {
    "instruction": ("You are a careful math tutor. Read the problem, reason step by step, "
                    "double-check the arithmetic, then give the final numeric answer on a new "
                    "line in the format '#### <answer>'.\n\n"),
    "demo_ids": (0, 1),
}

r1_dev, r1_records = evaluate(CFG_R1, dev, desc="R1 dev")
print(f"R1 (human few-shot) dev accuracy: {r1_dev:.2%}   (R0 was {r0_dev:.2%})")
rsi_dev["R1 human harness"] = r1_dev

## 6. Round 2 — *Agent-in-the-loop*: an LLM improves the harness for you

Now an **improver LLM** takes over the job you just did by hand. Each iteration it sees
(1) the current best config, (2) the recent attempts and scores, (3) a few dev failures —
and proposes one new config. Same budget-constrained loop, but the *analysis and the rewrite
are done by an agent*, not by you: that is the RSI step.

**Set up the improver API** (free tier is plenty — it is called only ~`MAX_EVALS` times):
register a key at DeepSeek / Groq / Gemini, click the **key icon** (Secrets) in Colab,
add a secret named `IMPROVER_API_KEY`, paste the key, and pick the provider below.

In [ ]:
import json, os

PROVIDERS = {   # pick one; key from Colab Secrets or env var IMPROVER_API_KEY
    "deepseek": ("https://api.deepseek.com",                    "deepseek-chat"),
    "groq":     ("https://api.groq.com/openai/v1",              "llama-3.1-8b-instant"),
    "gemini":   ("https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash"),
}
PROVIDER = "deepseek"   # <- change to "groq" or "gemini" if you prefer

try:
    from google.colab import userdata
    IMPROVER_API_KEY = userdata.get("IMPROVER_API_KEY")
except Exception:
    IMPROVER_API_KEY = os.environ.get("IMPROVER_API_KEY", "")
assert IMPROVER_API_KEY, ("No IMPROVER_API_KEY found (Colab Secrets or environment variable) -- "
                          "Track C requires a free-tier improver API. "
                          "(Track A of the assignment works without one.)")

from openai import OpenAI
IMPROVER_BASE, IMPROVER_MODEL = PROVIDERS[PROVIDER]
improver = OpenAI(api_key=IMPROVER_API_KEY, base_url=IMPROVER_BASE)
print("improver ready:", PROVIDER, IMPROVER_MODEL)


In [ ]:
OPTIMIZER_META_PROMPT = """You optimize prompts for a small LLM solving GSM8K math problems.
The target model reasons step by step and must end with '#### <number>'.

Current best configuration:
{config}

Recent attempts and their dev-subset accuracy:
{history}

A few problems the current best configuration still FAILS (model output -> gold answer):
{failures}

Demo pool: 20 worked examples (indices 0-19) that can be shown as few-shot demonstrations.

Propose ONE new configuration that beats the current best. Respond with JSON only, no extra text:
{{"instruction": "<one or two sentences>", "demo_ids": [<0 to 3 distinct indices from 0-19>]}}"""

def propose_config_llm(cur_cfg, history_txt, failures_txt):
    resp = improver.chat.completions.create(
        model=IMPROVER_MODEL, temperature=1.0,
        messages=[{"role": "user", "content": OPTIMIZER_META_PROMPT.format(
            config=json.dumps({"instruction": cur_cfg["instruction"], "demo_ids": list(cur_cfg["demo_ids"])}),
            history=history_txt or "(none yet)", failures=failures_txt)}])
    txt = resp.choices[0].message.content
    m = re.search(r"\{.*\}", txt, re.S)          # tolerate markdown fences etc.
    cfg = json.loads(m.group(0))
    demo_ids = tuple(sorted({int(d) for d in cfg.get("demo_ids", []) if 0 <= int(d) < DEMO_POOL_SIZE}))[:3]
    return {"instruction": str(cfg.get("instruction", INSTR))[:400], "demo_ids": demo_ids}

In [ ]:
SEARCH_DEV = dev.select(range(20))   # fast proxy set during the search
MAX_EVALS  = 8                       # budget: # of (LLM proposal + evaluation) pairs

best_cfg = max([CFG_R0, CFG_R1], key=lambda c: rsi_dev["R0 zero-shot"] if c is CFG_R0 else rsi_dev["R1 human harness"])
best_acc, search_log = -1.0, []

for step in range(MAX_EVALS):
    history_txt = "\n".join(
        f"- {json.dumps({'instruction': l['instruction'][:80], 'demos': l['k_demos']})} -> {l['dev_subset_acc']:.0%}"
        for l in search_log[-5:])
    fails = [r for r in (r1_records if best_cfg is CFG_R1 else r0_records) if not r["correct"]][:3]
    failures_txt = "\n".join(
        f"- Q: {r['question'][:200]}\n  model said: {str(r['pred'])}\n  gold: {r['gold']}" for r in fails)
    try:
        cfg = propose_config_llm(best_cfg, history_txt, failures_txt)
    except Exception as e:
        print("  (proposer call failed, retrying next step:", str(e)[:80], ")")
        continue
    acc, _ = evaluate(cfg, SEARCH_DEV, batch_size=20, max_new_tokens=200,
                      desc=f"try {step + 1}/{MAX_EVALS}")
    search_log.append({**cfg, "k_demos": len(cfg["demo_ids"]), "dev_subset_acc": acc})
    if acc > best_acc:
        best_cfg, best_acc = cfg, acc
    print(f"[{step + 1:>2}/{MAX_EVALS}] k_demos={len(cfg['demo_ids'])} "
          f"acc={acc:.2%}   best so far={best_acc:.2%}")

print("\nBest config found by the agent:", best_cfg)

In [ ]:
r2_dev, r2_records = evaluate(best_cfg, dev, desc="R2 dev")
print(f"R2 (agent-optimized harness) dev accuracy: {r2_dev:.2%}   "
      f"(R1 was {rsi_dev['R1 human harness']:.2%})")
rsi_dev["R2 agent harness"] = r2_dev

## 7. FINAL — one-shot evaluation on the hidden test set

**Run this once.** Report the numbers and do not go back to tune on them.

In [ ]:
rsi_test = {
    "R0 zero-shot":    evaluate(CFG_R0,  test, desc="test R0")[0],
    "R1 human harness": evaluate(CFG_R1, test, desc="test R1")[0],
    "R2 agent harness": evaluate(best_cfg, test, desc="test R2")[0],
}

print(f"{'Round':<18}{'dev':>8}{'test':>8}{'gap(dev-test)':>14}")
for k in rsi_dev:
    print(f"{k:<18}{rsi_dev[k]:>8.2%}{rsi_test[k]:>8.2%}{rsi_dev[k]-rsi_test[k]:>14.2%}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
ks = list(rsi_dev)
axes[0].plot(ks, [rsi_dev[k] for k in ks], "o-", label="dev (iterated on)")
axes[0].set_ylabel("accuracy"); axes[0].set_title("RSI curve (visible set)")
axes[0].tick_params(axis="x", rotation=20); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].bar(ks, [rsi_test[k] for k in ks], color="#3b7")
axes[1].set_ylabel("accuracy"); axes[1].set_title("ONE-SHOT hidden test")
axes[1].tick_params(axis="x", rotation=20); axes[1].set_ylim(0, 1)
plt.tight_layout(); plt.savefig("rsi_curve_trackC.png", dpi=150); plt.show()

## 8. What to put in your report

1. The table and curve from Section 7: dev trajectory, one-shot test score, generalization gap.
2. **Round 1**: which failures you saw, which demos/instruction you chose and why.
3. **Round 2**: the search log — what the agent proposed each step, what actually helped, and
   whether its reasoning about the failures was sound.
4. Your own rounds beyond R2 — at least two, each motivated by dev error analysis. Ideas:
   - more search budget (`MAX_EVALS`), better meta-prompt (show more failures, let it also
     propose decoding changes), a self-consistency variant of the best config;
   - a *verification* harness: ask the target model to re-check its arithmetic;
   - compare providers (DeepSeek vs Groq vs Gemini as the improver).

**Honesty section**: which improvements generalized to test, which only inflated dev.

## 9. Clean up

In [ ]:
import gc
gc.collect(); torch.cuda.empty_cache()
!nvidia-smi